# Rename the USGS observations: `usgs_inst_streamflow`, and add `usgs_daily_streamflow`

The v1 flows now write instantaneous (hourly) USGS streamflow to `usgs_inst_streamflow` and
daily means to a new `usgs_daily_streamflow` (`prefect-workflows/v1/flows/ingest_usgs.py`).
There's no rename in the control plane, so this registers the new datasets and moves what
refers to the old one:

| step | what |
| --- | --- |
| register | `usgs_inst_streamflow` (as `usgs_streamflow`: observed, inst, `approval_status`/`qualifier`, layer `usgs_gages`), `usgs_daily_streamflow` (mean, `day_starting`) and the derived `usgs_inst_streamflow_{percentiles,climatology,naive_scale}` |
| crosswalks | `usgs_nwm31` and `usgs_nwm30`: links saved to a Parquet file, the crosswalk deleted and registered again with `usgs_inst_streamflow` on the left, the links written back |

The old `usgs_streamflow`, its series table and derived datasets are left alone: deleting a
2B-row dataset can take the catalog down until teehr-cloud-core #274 is fixed.

**Run it as a platform admin, before deploying** the flows that use the new names (the old
flows keep working meanwhile: they read the crosswalks by name). Then deploy, and run
`v1-backfill-usgs-streamflow` and `v1-backfill-usgs-daily-streamflow`.

**Re-runnable:** registering skips datasets that exist; a crosswalk already on
`usgs_inst_streamflow` is left alone. Start with **2. Look**: it writes nothing.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import requests
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

## 1. Settings

In [ ]:
OLD, INST, DAILY = "usgs_streamflow", "usgs_inst_streamflow", "usgs_daily_streamflow"
#: Crosswalks whose left side moves from OLD to INST (their right side stays).
CROSSWALKS = ["usgs_nwm31", "usgs_nwm30"]
DERIVED = ["percentiles", "climatology", "naive_scale"]
#: Where each crosswalk's links are saved before it's deleted.
BACKUP = Path.home() / "usgs_rename_backup"

CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE  # KinD's certificate is self-signed
PUBLIC = "iceberg.public"

In [ ]:
# Local mode: the crosswalks are small.
spark = create_spark_session()
spark.conf.set("spark.sql.session.timeZone", "UTC")


def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:  # a kernel without the helper: the token as JupyterHub set it
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


api = requests.Session()
api.verify = VERIFY_TLS


def call(method, path, **kwargs):
    resp = api.request(method, f"{CONTROL_PLANE}/v1{path}",
                       headers={"Authorization": f"Bearer {access_token()}"}, **kwargs)
    if resp.status_code >= 400 and resp.status_code != 404:
        raise RuntimeError(f"{method} {path}: HTTP {resp.status_code}: {resp.text[:1000]}")
    return resp


def get(name):
    resp = call("GET", f"/projects/public/datasets/{name}")
    return resp.json() if resp.status_code == 200 else None


def register(body):
    if get(body["name"]):
        print(f"public.{body['name']}: exists")
        return
    call("POST", "/projects/public/datasets", json=body)
    print(f"public.{body['name']}: registered")


def links(name):
    spark.catalog.refreshTable(f"{PUBLIC}.{name}")
    return spark.table(f"{PUBLIC}.{name}").toPandas()


me = call("GET", "/projects/public").json()
print(f"public: {me['status']}; your role there: {me.get('my_role')}")

## 2. Look (writes nothing)

In [ ]:
old = get(OLD)
print(OLD, old["timeseries"])
for name in CROSSWALKS:
    record = get(name)
    print(name, record["crosswalk"] if record else "-", len(links(name)) if record else 0, "links")
for kind in DERIVED:
    record = get(f"{OLD}_{kind}")
    print(f"{OLD}_{kind}", record["derived"] if record else "-")
for name in (INST, DAILY, *(f"{INST}_{k}" for k in DERIVED)):
    print(name, "exists" if get(name) else "-")

## 3. Register the new datasets (as you)

`usgs_inst_streamflow` takes `usgs_streamflow`'s metadata (its own new series table).
`usgs_daily_streamflow`: daily means at midnight UTC of their day (`day_starting`), the same
optional columns and layer.

In [ ]:
FIELDS = ("role", "variable", "unit", "statistic", "period", "optional_columns",
          "location_layer", "time_partition", "source")
inst = {k: old["timeseries"][k] for k in FIELDS if old["timeseries"].get(k) is not None}
register({"type": "timeseries", "name": INST, **inst,
          "description": "USGS instantaneous streamflow (discharge, 00060), hourly values"})
register({
    "type": "timeseries", "name": DAILY, "role": "observed", "variable": "streamflow",
    "statistic": "mean", "period": "day_starting",
    "optional_columns": ["approval_status", "qualifier"],
    "location_layer": old["timeseries"]["location_layer"], "time_partition": "month",
    "source": "USGS Water Data (api.waterdata.usgs.gov): daily mean discharge, 00060, statistic 00003",
    "description": "USGS daily mean streamflow (discharge, 00060), at midnight UTC of each day",
})
for kind in DERIVED:
    register({"type": kind, "name": f"{INST}_{kind}", "observed": INST,
              "description": f"{kind.replace('_', ' ')} of {INST}"})

## 4. Move the crosswalks to `usgs_inst_streamflow`

For each: save its links (`BACKUP/<name>.parquet`), delete it, register it again with the
new left side (same name, right side, source and description), write the links back, and
check the count. Run it between NWM ingests: the `v1-ingest-nwm-*` flows read `usgs_nwm31`.

In [ ]:
BACKUP.mkdir(exist_ok=True)
for name in CROSSWALKS:
    record = get(name)
    if record and record["crosswalk"]["left"] in (INST, f"public.{INST}"):
        print(f"{name}: already on {INST}")
        continue
    saved = links(name)
    path = BACKUP / f"{name}.parquet"
    saved.to_parquet(path, index=False)
    print(f"{name}: {len(saved)} links saved to {path}")

    call("DELETE", f"/projects/public/datasets/{name}")
    meta = record["crosswalk"]
    register({"type": "crosswalk", "name": name, "left": INST, "right": meta["right"],
              **({"source": meta["source"]} if meta.get("source") else {}),
              **({"description": record["description"]} if record.get("description") else {})})
    spark.catalog.refreshTable(f"{PUBLIC}.{name}")
    columns = [f.name for f in spark.table(f"{PUBLIC}.{name}").schema]
    spark.createDataFrame(saved[columns]).writeTo(f"{PUBLIC}.{name}").append()
    print(f"{name}: {len(links(name))} links written back (saved {len(saved)})")

## 5. Check

In [ ]:
for name in CROSSWALKS:
    record = get(name)
    print(name, record["crosswalk"], len(links(name)), "links")
for name in (INST, DAILY, *(f"{INST}_{k}" for k in DERIVED)):
    record = get(name)
    print(name, record["kind"], record.get("timeseries") or record.get("derived"))